# Hafta 4: Karar Ağaçları (Decision Trees)

Bu notebook'ta aşağıdaki konular ele alınmaktadır:
1. Entropi ve Bilgi Kazancı hesaplama
2. Gini İndeksi hesaplama
3. Scikit-learn ile Karar Ağacı modeli oluşturma
4. Karar Ağacını görselleştirme
5. Model değerlendirme (Confusion Matrix, Accuracy, Precision, Recall, F1)
6. Öznitelik önemi (Feature Importance)
7. Budama (Pruning) ve Cross-Validation

## 1. Gerekli Kütüphanelerin Yüklenmesi

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.preprocessing import LabelEncoder

matplotlib.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', 20)
print("Kütüphaneler başarıyla yüklendi!")

## 2. Teorik Arka Plan: Entropi ve Bilgi Kazancı

### Entropi (Entropy)

Veri setindeki **belirsizliği** ölçer:

$$Entropy(S) = -\sum_{i} p_i \cdot \log_2(p_i)$$

- Tüm örnekler aynı sınıftan → Entropy = 0 (tam düzen)
- Örnekler eşit dağılmış → Entropy maksimum

### Bilgi Kazancı (Information Gain)

$$Gain(S,A) = Entropy(S) - \sum_{v \in Values(A)} \frac{|S_v|}{|S|} \cdot Entropy(S_v)$$

### Gini İndeksi

$$Gini(S) = 1 - \sum_{i} p_i^2$$

Düşük Gini → daha saf bölme

In [ ]:
# ---------------------------------------------------------------
# Entropi ve Bilgi Kazancı - Manuel Hesaplama
# PlayTennis klasik örneği
# ---------------------------------------------------------------

def entropy(probabilities):
    """Verilen olasılıklar için entropi hesaplar."""
    probs = np.array([p for p in probabilities if p > 0])
    return -np.sum(probs * np.log2(probs))

def information_gain(parent_probs, children):
    """
    parent_probs : [p_pos, p_neg] – tüm dataset
    children     : list of (weight, [p_pos, p_neg]) – alt kümeler
    """
    parent_entropy = entropy(parent_probs)
    weighted_entropy = sum(w * entropy(probs) for w, probs in children)
    return parent_entropy - weighted_entropy

def gini(probabilities):
    """Gini indeksi hesaplar."""
    return 1 - sum(p**2 for p in probabilities)

# PlayTennis veri seti özeti:
# Toplam: 9 Yes, 5 No  → n=14
total_yes, total_no, total = 9, 5, 14
p_yes = total_yes / total
p_no  = total_no  / total

S_entropy = entropy([p_yes, p_no])
S_gini    = gini([p_yes, p_no])

print("=" * 50)
print("PlayTennis Veri Seti Özeti")
print("=" * 50)
print(f"  Toplam örnek : {total}  (9 Yes, 5 No)")
print(f"  Entropy(S)   : {S_entropy:.4f}")
print(f"  Gini(S)      : {S_gini:.4f}")

# Outlook özniteliği için Bilgi Kazancı
# Sunny   : 2 Yes, 3 No  → 5 örnek
# Overcast: 4 Yes, 0 No  → 4 örnek
# Rain    : 3 Yes, 2 No  → 5 örnek
outlook_children = [
    (5/14,  [2/5,  3/5]),   # Sunny
    (4/14,  [4/4,  0/4]),   # Overcast  (saf düğüm)
    (5/14,  [3/5,  2/5]),   # Rain
]
gain_outlook = information_gain([p_yes, p_no], outlook_children)

print("\n--- Outlook Özniteliği ---")
print(f"  Sunny    → Entropy: {entropy([2/5, 3/5]):.4f}")
print(f"  Overcast → Entropy: {entropy([1.0]):.4f}")
print(f"  Rain     → Entropy: {entropy([3/5, 2/5]):.4f}")
print(f"  Gain(S, Outlook) = {gain_outlook:.4f}")

In [ ]:
# Entropi ve Gini eğrilerini görselleştir
p_values = np.linspace(0.001, 0.999, 200)
entropy_values = [-p * np.log2(p) - (1-p) * np.log2(1-p) for p in p_values]
gini_values    = [2 * p * (1-p) for p in p_values]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(p_values, entropy_values, 'b-', linewidth=2, label='Entropy')
axes[0].set_xlabel('p (pozitif sınıf oranı)')
axes[0].set_ylabel('Değer')
axes[0].set_title('Binary Entropy')
axes[0].axvline(0.5, color='gray', linestyle='--', alpha=0.6)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(p_values, gini_values, 'r-', linewidth=2, label='Gini')
axes[1].set_xlabel('p (pozitif sınıf oranı)')
axes[1].set_ylabel('Değer')
axes[1].set_title('Gini İndeksi')
axes[1].axvline(0.5, color='gray', linestyle='--', alpha=0.6)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Belirsizlik Ölçütleri', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Veri Seti Yükleme ve Keşif

**Diabetes veri seti** kullanılacaktır.  
Hedef değişken: `Outcome` — 1 = Diyabetik, 0 = Sağlıklı

In [ ]:
import os

# Veri seti yolu – notebook'un iki üst dizinindeki Veri_Setleri_Genel klasörü
DATA_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')),
                         '..', '..', 'Veri_Setleri_Genel', 'diabetes.csv')

df = pd.read_csv(DATA_PATH)
print(f"Veri seti boyutu: {df.shape[0]} satır, {df.shape[1]} sütun")
print("\nİlk 5 satır:")
df.head()

In [ ]:
print("Veri tipi ve eksik değer bilgisi:")
print(df.dtypes)
print("\nEksik değer sayıları:")
print(df.isnull().sum())
print("\nTemel istatistikler:")
df.describe()

In [ ]:
# Hedef değişken dağılımı
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Sol: sınıf dağılımı
counts = df['Outcome'].value_counts()
axes[0].bar(['Sağlıklı (0)', 'Diyabetik (1)'], counts.values,
            color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Hedef Değişken Dağılımı')
axes[0].set_ylabel('Örnek Sayısı')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Sağ: özniteliklerin korelasyon ısı haritası
corr = df.corr()
mask = np.zeros_like(corr)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            ax=axes[1], mask=mask, linewidths=0.5)
axes[1].set_title('Korelasyon Matrisi')

plt.tight_layout()
plt.show()

## 4. Veri Ön İşleme ve Eğitim / Test Ayrımı

In [ ]:
# Öznitelikler ve hedef değişken
feature_cols = [c for c in df.columns if c != 'Outcome']
X = df[feature_cols].values
y = df['Outcome'].values

# Eğitim / Test ayrımı: %80 eğitim, %20 test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Toplam örnek  : {len(X)}")
print(f"Eğitim seti   : {len(X_train)}  ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test seti     : {len(X_test)}   ({len(X_test)/len(X)*100:.1f}%)")
print(f"\nEğitim – sınıf dağılımı : {np.bincount(y_train)}")
print(f"Test     – sınıf dağılımı : {np.bincount(y_test)}")

## 5. Karar Ağacı Modeli Oluşturma

İki farklı kriter ile model eğitip karşılaştırıyoruz:
- **Gini İndeksi** (CART algoritması – scikit-learn varsayılanı)
- **Bilgi Kazancı / Entropi** (ID3 / C4.5 benzeri)

In [ ]:
# Model 1: Gini kriteri (CART)
clf_gini = DecisionTreeClassifier(criterion='gini', max_depth=5, random_state=42)
clf_gini.fit(X_train, y_train)

# Model 2: Entropy / Bilgi Kazancı kriteri
clf_entropy = DecisionTreeClassifier(criterion='entropy', max_depth=5, random_state=42)
clf_entropy.fit(X_train, y_train)

# Performans karşılaştırması
models = {'Gini': clf_gini, 'Entropy': clf_entropy}
print(f"{'Kriter':<12} {'Eğitim Doğruluğu':>18} {'Test Doğruluğu':>16}")
print("-" * 48)
for name, model in models.items():
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc  = accuracy_score(y_test,  model.predict(X_test))
    print(f"{name:<12} {train_acc:>17.4f} {test_acc:>15.4f}")

## 6. Karar Ağacını Görselleştirme

In [ ]:
# Gini modeli karar ağacını görselleştir (max_depth=3 ile daha okunabilir)
clf_vis = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=42)
clf_vis.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(22, 8))
plot_tree(
    clf_vis,
    feature_names=feature_cols,
    class_names=['Sağlıklı', 'Diyabetik'],
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax
)
plt.title('Karar Ağacı (Gini, max_depth=3)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Metin tabanlı ağaç gösterimi
print("\n--- Metin Tabanlı Ağaç ---")
print(export_text(clf_vis, feature_names=feature_cols))

## 7. Model Değerlendirme

### Karmaşıklık Matrisi (Confusion Matrix)

$$\begin{array}{c|cc} & \text{Tahmin: Pos} & \text{Tahmin: Neg} \\ \hline \text{Gerçek: Pos} & TP & FN \\ \text{Gerçek: Neg} & FP & TN \end{array}$$

### Metrikler

$$Accuracy = \frac{TP + TN}{Total} \qquad Precision = \frac{TP}{TP + FP}$$

$$Recall = \frac{TP}{TP + FN} \qquad F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}$$

In [ ]:
# Ana model: Gini, max_depth=5
y_pred = clf_gini.predict(X_test)

# 1) Karmaşıklık Matrisi
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Sağlıklı (0)', 'Diyabetik (1)'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Karmaşıklık Matrisi (Gini, depth=5)')

# 2) Metrik özeti
from sklearn.metrics import precision_score, recall_score, f1_score

metrics = {
    'Accuracy' : accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall'   : recall_score(y_test, y_pred),
    'F1-Score' : f1_score(y_test, y_pred),
}
axes[1].barh(list(metrics.keys()), list(metrics.values()),
             color=['steelblue', 'seagreen', 'tomato', 'darkorange'])
axes[1].set_xlim(0, 1.1)
axes[1].set_title('Değerlendirme Metrikleri')
for i, (k, v) in enumerate(metrics.items()):
    axes[1].text(v + 0.01, i, f'{v:.4f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

print("\n--- Sınıflandırma Raporu ---")
print(classification_report(y_test, y_pred,
                             target_names=['Sağlıklı', 'Diyabetik']))

## 8. Öznitelik Önemi (Feature Importance)

In [ ]:
# Öznitelik önemi
importances = clf_gini.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 5))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(feature_cols)))
plt.bar(range(len(feature_cols)),
        importances[indices],
        color=colors, edgecolor='black')
plt.xticks(range(len(feature_cols)),
           [feature_cols[i] for i in indices],
           rotation=35, ha='right')
plt.title('Öznitelik Önemi (Gini, depth=5)', fontsize=13, fontweight='bold')
plt.ylabel('Önem Skoru')
plt.tight_layout()
plt.show()

print("\nÖnem sıralaması:")
for i in indices:
    print(f"  {feature_cols[i]:<25} : {importances[i]:.4f}")

## 9. Budama (Pruning) ve Hiperparametre Optimizasyonu

Aşırı öğrenmeyi önlemek için:
- **Pre-pruning:** `max_depth`, `min_samples_split`, `min_samples_leaf`
- **Cost-Complexity Pruning:** `ccp_alpha` parametresi ile post-pruning

In [ ]:
# max_depth değiştirildiğinde eğitim ve test doğruluğunun değişimi
depths = range(1, 21)
train_scores, test_scores = [], []

for d in depths:
    m = DecisionTreeClassifier(criterion='gini', max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, m.predict(X_train)))
    test_scores.append(accuracy_score(y_test,  m.predict(X_test)))

plt.figure(figsize=(10, 5))
plt.plot(depths, train_scores, 'b-o', label='Eğitim Doğruluğu', markersize=4)
plt.plot(depths,  test_scores, 'r-s', label='Test Doğruluğu',   markersize=4)
plt.axvline(5, color='gray', linestyle='--', alpha=0.7, label='max_depth=5')
plt.xlabel('max_depth')
plt.ylabel('Doğruluk (Accuracy)')
plt.title('max_depth ve Eğitim/Test Doğruluğu İlişkisi', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_depth = depths[np.argmax(test_scores)]
print(f"En iyi test doğruluğu → max_depth = {best_depth}  ({max(test_scores):.4f})")

In [ ]:
# 5-Katlı Cross-Validation
clf_cv = DecisionTreeClassifier(criterion='gini', max_depth=5, random_state=42)
cv_scores = cross_val_score(clf_cv, X, y, cv=5, scoring='accuracy')

print("5-Katlı Çapraz Doğrulama Sonuçları:")
for i, s in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {s:.4f}")
print(f"\n  Ortalama : {cv_scores.mean():.4f}")
print(f"  Std Dev  : {cv_scores.std():.4f}")

In [ ]:
# GridSearchCV ile en iyi hiperparametreleri arama
param_grid = {
    'max_depth'        : [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 5],
    'criterion'        : ['gini', 'entropy'],
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

print("En iyi parametreler:")
for k, v in grid_search.best_params_.items():
    print(f"  {k:<22} : {v}")

best_model = grid_search.best_estimator_
best_test_acc = accuracy_score(y_test, best_model.predict(X_test))
print(f"\nCV Ortalama Doğruluk  : {grid_search.best_score_:.4f}")
print(f"Test Doğruluğu        : {best_test_acc:.4f}")

In [ ]:
# Cost-Complexity Pruning (Post-pruning) – ccp_alpha
path = DecisionTreeClassifier(random_state=42).cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas[:-1]  # Son eleman (köksüz ağaç) çıkarılır

train_acc_list, test_acc_list = [], []
for alpha in ccp_alphas:
    clf_p = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
    clf_p.fit(X_train, y_train)
    train_acc_list.append(accuracy_score(y_train, clf_p.predict(X_train)))
    test_acc_list.append(accuracy_score(y_test,   clf_p.predict(X_test)))

plt.figure(figsize=(10, 5))
plt.plot(ccp_alphas, train_acc_list, marker='o', markersize=3,
         label='Eğitim Doğruluğu', color='steelblue')
plt.plot(ccp_alphas,  test_acc_list, marker='s', markersize=3,
         label='Test Doğruluğu',   color='tomato')
plt.xlabel('ccp_alpha')
plt.ylabel('Doğruluk')
plt.title('Cost-Complexity Pruning (Post-Pruning)', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_alpha = ccp_alphas[np.argmax(test_acc_list)]
print(f"En iyi ccp_alpha : {best_alpha:.6f}  →  Test Doğruluğu: {max(test_acc_list):.4f}")

## 10. Özet

| Konu | Öğrenilenler |
|------|-------------|
| **Entropi** | Veri setindeki belirsizliği ölçer; $H = -\sum p_i \log_2 p_i$ |
| **Bilgi Kazancı** | En iyi özniteliği seçer (ID3 / C4.5) |
| **Gini İndeksi** | Alternatif safsızlık ölçütü (CART); daha hızlı |
| **Karar Ağacı** | `DecisionTreeClassifier(criterion, max_depth)` |
| **Görselleştirme** | `plot_tree()` ve `export_text()` |
| **Değerlendirme** | Accuracy, Precision, Recall, F1, Confusion Matrix |
| **Öznitelik Önemi** | `feature_importances_` |
| **Budama** | Pre-pruning (`max_depth`) ve Post-pruning (`ccp_alpha`) |
| **Cross-Validation** | `cross_val_score()` ile güvenilir değerlendirme |
| **GridSearchCV** | Hiperparametre arama |

**Bir sonraki hafta:** Kural Çıkarımı ve k-NN Algoritması